# A/B Testing with Amazon Bedrock AgentCore (Windows)

This notebook demonstrates **target-based A/B testing** using Amazon Bedrock AgentCore.

**Prerequisites:** Python 3.12+, `uv`, Node.js, AWS CLI >= 2.34, CDK bootstrapped, Bedrock model access.

**Run this notebook from the `ab_testing/` directory.**

## 1. Prerequisites Check

In [ ]:
import subprocess, os, sys, platform, json, time, uuid, glob

def run_cmd(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True, shell=True, encoding='utf-8')
    if r.returncode != 0:
        raise RuntimeError(f'Command failed: {cmd}\n{r.stderr}')
    return r.stdout.strip()

APP_NAME = 'fixFirstAgent'
REGION = run_cmd('aws configure get region') or 'us-east-1'

def get_param(name):
    return run_cmd(f'aws ssm get-parameter --name /{APP_NAME}/{name} --query Parameter.Value --output text --region {REGION}')

TARGET_DIR = os.path.join(os.getcwd(), 'target_based_variants')
AB_CDK_DIR = os.path.join(TARGET_DIR, 'cdk_ab_testing')
AGENTS_DIR = os.path.join(TARGET_DIR, 'agents')

proc = subprocess.run(
    os.path.join(os.getcwd(), 'win', 'check_prerequisites.bat'),
    capture_output=True, text=True, shell=True, encoding='utf-8'
)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)

## 2. Package Agents

In [ ]:
proc = subprocess.run(
    os.path.join(os.getcwd(), 'win', 'package_agents.bat') + f' "{AGENTS_DIR}"',
    capture_output=True, text=True, shell=True, encoding='utf-8'
)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError('Agent packaging failed')
print('Both agents packaged')

## 3. Deploy Runtimes + Evaluation Configs

In [ ]:
proc = subprocess.Popen(
    'npx cdk deploy fixFirstAgent-ABTestingStack --require-approval never',
    cwd=AB_CDK_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    shell=True
)
raw_output, _ = proc.communicate()
output = raw_output.decode('utf-8', errors='replace')
print(output[-3000:] if len(output) > 3000 else output)
if proc.returncode != 0:
    raise RuntimeError('CDK deploy failed')
print('Runtime + Eval stack deployed')

## 4. Wait for Runtimes to Become READY

In [ ]:
CONTROL_RUNTIME_ARN = get_param('control-runtime-arn')
REFINED_RUNTIME_ARN = get_param('refined-runtime-arn')
CONTROL_RUNTIME_ID = CONTROL_RUNTIME_ARN.split('/')[-1]
REFINED_RUNTIME_ID = REFINED_RUNTIME_ARN.split('/')[-1]

print(f'Control: {CONTROL_RUNTIME_ARN}')
print(f'Treatment: {REFINED_RUNTIME_ARN}')

for name, rid in [('Control', CONTROL_RUNTIME_ID), ('Treatment', REFINED_RUNTIME_ID)]:
    print(f'\nWaiting for {name} runtime...')
    for i in range(30):
        status = run_cmd(f'aws bedrock-agentcore-control get-agent-runtime --agent-runtime-id {rid} --region {REGION} --query status --output text')
        if status == 'READY':
            print(f'  {name} is READY')
            break
        print(f'  [{i+1}/30] {status}')
        time.sleep(20)
    else:
        raise TimeoutError(f'{name} not READY: {status}')

## 5. Send Requests to Each Agent (Generate Evaluation Data)

In [ ]:
with open('prompts.txt') as f:
    prompts = [line.strip() for line in f if line.strip()][:10]

for name, arn in [('Control', CONTROL_RUNTIME_ARN), ('Treatment', REFINED_RUNTIME_ARN)]:
    print(f'\n=== {name} Agent ===')
    for i, prompt in enumerate(prompts):
        sid = f'eval-{name.lower()}-{uuid.uuid4()}'
        with open('_payload.json', 'w') as pf:
            json.dump({'prompt': prompt}, pf)
        run_cmd(
            f'aws bedrock-agentcore invoke-agent-runtime'
            f' --agent-runtime-arn {arn}'
            f' --qualifier DEFAULT'
            f' --runtime-session-id {sid}'
            f' --payload fileb://_payload.json'
            f' --region {REGION}'
            f' _response.json'
        )
        print(f'  [{i+1}/10] {prompt[:50]}')
    print(f'{name}: 10 requests sent')

for f in glob.glob('_payload.json') + glob.glob('_response.json'):
    os.remove(f)
print('\nAll requests sent. Evaluation scores will appear in ~15 minutes.')

## 6. Deploy Gateway + Targets + A/B Test

In [ ]:
CONTROL_EVAL_ARN = get_param('control-eval-arn')
TREATMENT_EVAL_ARN = get_param('treatment-eval-arn')

AB_GW_CDK_DIR = os.path.join(TARGET_DIR, 'cdk_ab_gateway')
deploy_cmd = (
    f'npx cdk deploy fixFirstAgent-ABGatewayStack --require-approval never'
    f' -c controlRuntimeArn={CONTROL_RUNTIME_ARN}'
    f' -c refinedRuntimeArn={REFINED_RUNTIME_ARN}'
    f' -c controlEvalArn={CONTROL_EVAL_ARN}'
    f' -c treatmentEvalArn={TREATMENT_EVAL_ARN}'
)
proc = subprocess.Popen(
    deploy_cmd, cwd=AB_GW_CDK_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, shell=True
)
raw_output, _ = proc.communicate()
output = raw_output.decode('utf-8', errors='replace')
print(output[-3000:] if len(output) > 3000 else output)
if proc.returncode != 0:
    raise RuntimeError('CDK deploy failed')
print('Gateway stack deployed')

## 7. Send Traffic Through Gateway

In [ ]:
GATEWAY_URL = get_param('ab-gateway-url')
print(f'Gateway URL: {GATEWAY_URL}')

scripts_dir = os.path.join(os.getcwd(), 'scripts')
proc = subprocess.run(
    f'python "{os.path.join(scripts_dir, "send_traffic.py")}" "{GATEWAY_URL}" "{REGION}" prompts.txt',
    capture_output=True, text=True, shell=True, encoding='utf-8'
)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError('Send traffic failed')

## 8. Check A/B Test Results

Re-run this cell after ~15 minutes.

- `p-value < 0.05` + positive change = treatment is significantly better
- `p-value >= 0.05` = not enough evidence yet

In [ ]:
AB_TEST_ID = get_param('ab-test-id')
print(f'A/B Test: {AB_TEST_ID}')

result = json.loads(run_cmd(
    f'aws bedrock-agentcore get-ab-test --ab-test-id {AB_TEST_ID} --region {REGION} --output json'
))
print(f'Status: {result["status"]}')
print(f'Execution: {result["executionStatus"]}')

results = result.get('results')
if results:
    print(f'\nAnalysis timestamp: {results.get("analysisTimestamp")}')
    print('=' * 60)
    for metric in results['evaluatorMetrics']:
        evaluator = metric.get('evaluatorArn', 'Unknown').split('/')[-1]
        control = metric['controlStats']
        print(f'\nEvaluator: {evaluator}')
        print(f'  Control (C):   mean={control["mean"]:.3f}, samples={control["sampleSize"]}')
        for v in metric['variantResults']:
            sig = 'YES' if v['isSignificant'] else 'no'
            print(f'  Treatment (T1): mean={v["mean"]:.3f}, samples={v["sampleSize"]}')
            print(f'    Change: {v.get("percentChange", 0):.1f}%')
            print(f'    p-value: {v.get("pValue", "N/A")}')
            print(f'    Significant: {sig}')
            if v.get('confidenceInterval'):
                ci = v['confidenceInterval']
                print(f'    95% CI: [{ci["lower"]:.3f}, {ci["upper"]:.3f}]')
    print('\n' + '=' * 60)
    all_sig = all(v['isSignificant'] for m in results['evaluatorMetrics'] for v in m['variantResults'])
    if all_sig:
        winner_change = results['evaluatorMetrics'][0]['variantResults'][0].get('percentChange', 0)
        if winner_change > 0:
            print('RECOMMENDATION: Treatment (T1) is significantly better. Consider deploying it.')
        else:
            print('RECOMMENDATION: Control (C) is significantly better. Keep the current agent.')
    else:
        print('RECOMMENDATION: Not yet significant. Continue collecting samples or increase traffic.')
else:
    print('\nNo results yet.')
    print('Results appear after sessions complete (~15 min after session timeout).')
    print('Re-run this cell later to check for updates.')

## 9. Stop the A/B Test

In [ ]:
AB_TEST_ID = get_param('ab-test-id')
run_cmd(f'aws bedrock-agentcore update-ab-test --ab-test-id {AB_TEST_ID} --execution-status STOPPED --region {REGION}')
print(f'A/B test {AB_TEST_ID} stopped')

## 10. Cleanup

In [ ]:
proc = subprocess.run(
    f'python "{os.path.join(TARGET_DIR, "scripts", "cleanup_ab_test.py")}"',
    capture_output=True, text=True, shell=True, encoding='utf-8',
    env={**os.environ, 'AWS_REGION': REGION, 'APP_NAME': APP_NAME}
)
print(proc.stdout)
if proc.stderr:
    print(proc.stderr)

for cdk_dir in [os.path.join(TARGET_DIR, 'cdk_ab_gateway'), AB_CDK_DIR]:
    stack = 'fixFirstAgent-ABGatewayStack' if 'gateway' in cdk_dir else 'fixFirstAgent-ABTestingStack'
    p = subprocess.Popen(
        f'npx cdk destroy {stack} --force',
        cwd=cdk_dir, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, shell=True
    )
    out, _ = p.communicate()
    print(out.decode('utf-8', errors='replace'))

print('All A/B testing infrastructure destroyed.')